# Практическое задание: отбор признаков и понижение размерности

**Уровень:** начинающий  **Основа:** лекция «Понижение размерности и отбор признаков» (уроки 3.1–3.7)

**Данные:** *Breast Cancer Wisconsin (Diagnostic)* — реальные медицинские данные о 569 пациентках. По 30 числовым характеристикам клеточных ядер (радиус, текстура, периметр, площадь, гладкость и т.д.), полученным из изображений биопсии, нужно определить, опухоль **злокачественная** или **доброкачественная**. Датасет встроен в `scikit-learn`, скачивать ничего не нужно.

**Как выполнять.** Ячейки с пометкой `# ВАШ КОД` нужно дописать. Вместо `...` подставьте свой код. Ответы на вопросы пишите в markdown-ячейках «Ваш ответ». После каждой задачи (кроме первой) есть ячейка **«Проверка»** с `assert`. Если она выполнилась без ошибок и напечатала ✅, задача решена верно. Если вы видите `AssertionError`, прочитайте сообщение: оно подскажет, что не так. Перед сдачей выполните *Kernel → Restart & Run All*, чтобы убедиться, что ноутбук работает от начала до конца.

| № | Тема (раздел лекции) | Баллы |
|---|---|---|
| 1 | Знакомство с данными | 5 |
| 2 | Зачем отбирать признаки: шумовые признаки (3.1) | 10 |
| 3 | Одномерный отбор: корреляция, ROC-AUC, взаимная информация (3.2) | 15 |
| 4 | Проблема одномерного подхода: зависимые признаки (3.2.5, 3.5) | 10 |
| 5 | Жадное добавление признаков (3.3) | 15 |
| 6 | Отбор на основе моделей: L1, случайный лес, перестановки (3.4) | 15 |
| 7 | Метод главных компонент (3.6–3.7) | 15 |
| 8 | Случайные проекции (3.5.2) | 10 |
| 9 | Итоговое сравнение и выводы | 5 |
| | **Итого** | **100** |

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA
from sklearn.random_projection import GaussianRandomProjection
from scipy.spatial.distance import pdist

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 4)
RANDOM_STATE = 42

## Задача 1. Знакомство с данными (5 баллов)

Загрузим датасет. В `y` значение **0** означает злокачественную опухоль, **1** — доброкачественную.

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target
print(data.target_names)  # ['malignant' 'benign']

**1.1.** Выведите размер таблицы `X`, число объектов каждого класса и сводную статистику (`mean`, `std`, `min`, `max`) для первых 10 признаков.

In [ ]:
# ВАШ КОД
print("Объектов и признаков:", ...)
...  # число объектов каждого класса
...  # сводная статистика по первым 10 признакам

**1.2.** Ответьте: сбалансированы ли классы? Почему перед обучением линейных моделей важно масштабировать признаки (подсказка: раздел 3.4.1)?

**Ваш ответ:** *(двойным щелчком откройте ячейку и впишите ответ)*

Разобьём данные на обучающую и тестовую выборки и масштабируем признаки. Масштабатор обучается **только на обучающей выборке**. Эту ячейку менять не нужно.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)

scaler = StandardScaler().fit(X_train)
X_train_s = pd.DataFrame(scaler.transform(X_train), columns=X.columns, index=X_train.index)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)


def evaluate(Xtr, Xte, model=None):
    """Обучает модель на Xtr и возвращает (accuracy, ROC-AUC) на Xte."""
    if model is None:
        model = LogisticRegression(max_iter=1000)
    model.fit(Xtr, y_train)
    proba = model.predict_proba(Xte)[:, 1]
    return round(accuracy_score(y_test, model.predict(Xte)), 4), round(roc_auc_score(y_test, proba), 4)


results = {}  # сюда будем складывать итоги: {название метода: (число признаков, accuracy, AUC)}
acc, auc = evaluate(X_train_s, X_test_s)
results["Все 30 признаков"] = (30, acc, auc)
print("Базовая модель на всех признаках: accuracy =", acc, ", AUC =", auc)

## Задача 2. Зачем отбирать признаки: шумовые признаки (10 баллов)

В разделе 3.1.1 объясняется, что шумовые признаки вредят модели. Проверим это экспериментально: добавим к данным **100 случайных признаков** из стандартного нормального распределения, не связанных с целевой переменной.

In [ ]:
rng = np.random.RandomState(RANDOM_STATE)
noise_cols = [f"noise_{i}" for i in range(100)]
noise_train = pd.DataFrame(rng.normal(size=(len(X_train_s), 100)), columns=noise_cols, index=X_train_s.index)
noise_test = pd.DataFrame(rng.normal(size=(len(X_test_s), 100)), columns=noise_cols, index=X_test_s.index)

X_train_noisy = pd.concat([X_train_s, noise_train], axis=1)
X_test_noisy = pd.concat([X_test_s, noise_test], axis=1)
print(X_train_noisy.shape)

**2.1.** Сравните качество (accuracy и AUC на тесте) логистической регрессии и kNN (`KNeighborsClassifier(n_neighbors=5)`) на исходных 30 признаках и на 130 признаках с шумом.

In [ ]:
# ВАШ КОД
noise_results = {}
for name, make_model in [("LogReg", lambda: LogisticRegression(max_iter=1000)),
                         ("kNN", lambda: KNeighborsClassifier(n_neighbors=5))]:
    clean = ...   # evaluate(...) на исходных признаках
    noisy = ...   # evaluate(...) на признаках с шумом
    noise_results[name] = (clean, noisy)
    print(f"{name}: чистые {clean}, с шумом {noisy}")

**2.2.** Найдите максимальную по модулю корреляцию с целевой переменной среди 100 шумовых признаков **на обучающей выборке** и такую же корреляцию на **тестовой** для того же признака. Что вы видите? Свяжите с первым примером из раздела 3.1.1.

In [ ]:
# ВАШ КОД
corr_noise_train = noise_train.corrwith(y_train).abs()
worst = ...  # имя шумового признака с максимальной корреляцией на train
print("Признак:", worst)
print("Корреляция на train:", ...)
print("Корреляция на test: ", ...)

In [ ]:
# Проверка задачи 2
assert set(noise_results) == {"LogReg", "kNN"}, "В noise_results должны быть оба метода: LogReg и kNN"
for name, (clean, noisy) in noise_results.items():
    assert len(clean) == 2 and len(noisy) == 2, "evaluate возвращает пару (accuracy, AUC)"
assert noise_results["kNN"][1][0] < noise_results["kNN"][0][0],     "С шумом точность kNN должна упасть. Проверьте, что вы передали зашумлённые выборки"
assert worst in noise_cols, "worst должен быть именем шумового признака"
assert worst == noise_train.corrwith(y_train).abs().idxmax(), "worst — шумовой признак с максимальной |корреляцией| на train"
print("✅ Задача 2: проверка пройдена")

**Ваш ответ на 2.1–2.2:** *(как изменилось качество у каждой модели? Какая модель сильнее пострадала и почему? Что произошло с корреляцией шумового признака на тесте?)*

## Задача 3. Одномерный отбор признаков (15 баллов)

В разделе 3.2 описаны три способа оценить информативность каждого признака по отдельности. Применим их к 30 исходным признакам (обучающая выборка).

**3.1.** Посчитайте для каждого признака:
- `corr` — модуль корреляции с целевой переменной (формула 3.2.2);
- `auc` — качество «классификатора по одному признаку» (раздел 3.2.3): `roc_auc_score(y_train, признак)`. Так как признак может быть связан с классом в обратную сторону, берите `max(auc, 1 - auc)`;
- `mi` — взаимная информация (раздел 3.2.4): `mutual_info_classif(X_train_s, y_train, random_state=RANDOM_STATE)`.

Соберите результаты в таблицу `scores` (индекс — названия признаков).

In [ ]:
# ВАШ КОД
corr = ...
auc = ...
mi = ...
scores = pd.DataFrame({"corr": corr, "auc": auc, "mi": mi})
display(scores.sort_values("corr", ascending=False).head(10))

**3.2.** Выведите топ-5 признаков по каждому из трёх методов и постройте столбчатую диаграмму топ-10 по корреляции.

In [ ]:
# ВАШ КОД
for col in ["corr", "auc", "mi"]:
    print(col, ":", ...)  # список топ-5 признаков

...  # barh-диаграмма топ-10 по корреляции
plt.show()

**3.3.** Отберите топ-10 признаков по `corr` и обучите на них логистическую регрессию. Сохраните результат в `results`. Затем проверьте, насколько методы согласны между собой: сколько признаков общих у топ-10 по `corr` и топ-10 по `mi`?

In [ ]:
# ВАШ КОД
top10_corr = ...  # список названий
acc, auc_ = evaluate(X_train_s[top10_corr], X_test_s[top10_corr])
results["Топ-10 по корреляции"] = (10, acc, auc_)
print(acc, auc_)

top10_mi = ...
print("Общих признаков:", ...)

**3.4.** Вернёмся к данным с шумом из задачи 2. Посчитайте корреляцию с целевой переменной для всех 130 признаков и определите, сколько шумовых признаков попало в топ-30. Можно ли доверять одномерному отбору, если шумовых признаков много?

In [ ]:
# ВАШ КОД
corr_noisy = ...
top30 = ...
print("Шумовых признаков в топ-30:", ...)

In [ ]:
# Проверка задачи 3
assert scores.shape == (30, 3), "scores должна быть таблицей: 30 признаков × 3 метода"
assert scores["corr"].between(0, 1).all(), "corr — модуль корреляции, значения от 0 до 1"
assert scores["auc"].between(0.5, 1).all(), "auc нужно брать как max(auc, 1 - auc): значения от 0.5 до 1"
assert (scores["mi"] >= 0).all(), "взаимная информация не бывает отрицательной"
expected = abs(np.corrcoef(X_train_s["worst perimeter"], y_train)[0, 1])
assert abs(scores.loc["worst perimeter", "corr"] - expected) < 1e-6, "Неверно посчитана корреляция"
assert len(top10_corr) == 10 and top10_corr[0] == scores["corr"].idxmax(), "top10_corr: 10 лучших по corr, по убыванию"
assert len(top10_mi) == 10, "top10_mi: 10 лучших по mi"
assert "Топ-10 по корреляции" in results, "Добавьте результат в results"
assert len(corr_noisy) == 130 and len(top30) == 30, "Для шумных данных нужна корреляция всех 130 признаков и топ-30"
print("✅ Задача 3: проверка пройдена")

**Ваш ответ на 3.2–3.4:** *(совпадают ли топ-признаки у разных методов? Что показал эксперимент с шумом?)*

## Задача 4. Проблема одномерного подхода: зависимые признаки (10 баллов)

Одномерные методы оценивают признаки независимо друг от друга (раздел 3.2.5). В нашем датасете много признаков сильно связаны между собой: радиус, периметр и площадь ядра клетки описывают по сути одно и то же.

**4.1.** Постройте тепловую карту корреляций между первыми 10 признаками (`mean ...`) на обучающей выборке.

In [ ]:
# ВАШ КОД
corr_matrix = ...
plt.figure(figsize=(7, 6))
plt.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(10), corr_matrix.columns, rotation=90)
plt.yticks(range(10), corr_matrix.columns)
plt.show()

**4.2.** Найдите все пары из **всех 30** признаков, у которых модуль корреляции больше 0.95, и выведите 5 самых сильно связанных пар.

In [ ]:
# ВАШ КОД
cm = X_train_s.corr().abs()
upper = cm.where(np.triu(np.ones(cm.shape), k=1).astype(bool))  # верхний треугольник без диагонали
pairs = upper.stack().sort_values(ascending=False)
n_high = ...
print("Пар с |corr| > 0.95:", n_high)
display(pairs.head(5))

In [ ]:
# Проверка задачи 4
assert corr_matrix.shape == (10, 10), "Нужны корреляции только между первыми 10 признаками"
assert list(corr_matrix.columns) == list(X_train_s.columns[:10]), "Возьмите первые 10 столбцов X_train_s"
assert np.allclose(np.diag(corr_matrix), 1), "На диагонали корреляционной матрицы должны быть единицы"
assert n_high == int((pairs > 0.95).sum()), "Посчитайте число пар с |corr| > 0.95 в таблице pairs"
print("✅ Задача 4: проверка пройдена")

**4.3.** Посмотрите на топ-10 по корреляции из задачи 3.3. Есть ли среди них признаки, которые почти дублируют друг друга? Почему это плохо для отбора и какой подход (разделы 3.3 и 3.5) помогает с этим справиться?

**Ваш ответ:** *...*

## Задача 5. Жадное добавление признаков (15 баллов)

Жадный алгоритм (раздел 3.3.3): начинаем с пустого набора и на каждом шаге добавляем тот признак, который сильнее всего улучшает качество модели. Останавливаемся, когда улучшения нет. Качество оцениваем кросс-валидацией на обучающей выборке (метрика ROC-AUC, 5 фолдов), тест не трогаем.

**5.1.** Допишите функцию `greedy_forward`.

In [ ]:
def greedy_forward(X, y, max_features=10, cv=5):
    selected = []
    remaining = list(X.columns)
    best_score = -np.inf
    history = []

    while remaining and len(selected) < max_features:
        scores_step = {}
        for f in remaining:
            # ВАШ КОД: оцените кросс-валидацией модель на признаках selected + [f]
            scores_step[f] = ...

        best_f = max(scores_step, key=scores_step.get)
        if scores_step[best_f] <= best_score:   # улучшения нет, останавливаемся
            break

        # ВАШ КОД: добавьте best_f в selected, уберите из remaining,
        # обновите best_score и запишите (best_f, best_score) в history
        ...

    return selected, history

**5.2.** Запустите алгоритм на масштабированной обучающей выборке, выведите порядок добавления признаков и постройте график «число признаков — CV-AUC». Затем оцените итоговый набор на тесте и добавьте результат в `results`.

In [ ]:
# ВАШ КОД
selected_greedy, history = greedy_forward(X_train_s, y_train)
for i, (f, s) in enumerate(history, 1):
    print(i, f, round(s, 4))

...  # график зависимости CV-AUC от числа признаков
plt.show()

acc, auc_ = ...
results["Жадное добавление"] = (len(selected_greedy), acc, auc_)
print(acc, auc_)

In [ ]:
# Проверка задачи 5
assert 0 < len(selected_greedy) <= 10, "Должен быть выбран хотя бы один признак, но не больше 10"
assert len(set(selected_greedy)) == len(selected_greedy), "Признаки в selected не должны повторяться"
assert [h[0] for h in history] == selected_greedy, "history должна содержать пары (признак, CV-AUC) в порядке добавления"
cv_scores = [h[1] for h in history]
assert all(b > a for a, b in zip(cv_scores, cv_scores[1:])), "CV-AUC должна расти на каждом шаге (иначе алгоритм останавливается)"
assert cv_scores[-1] > 0.98, "Итоговый CV-AUC должен быть выше 0.98: проверьте оценку качества внутри цикла"
assert results["Жадное добавление"][0] == len(selected_greedy), "Добавьте результат в results"
print("✅ Задача 5: проверка пройдена")

**5.3.** Ответьте на вопросы:
1. Сколько всего подмножеств пришлось бы перебрать при полном переборе 30 признаков (раздел 3.3.2)? Сколько обучений модели сделал жадный алгоритм (приблизительно: число шагов × число кандидатов × 5 фолдов)?
2. Почему алгоритм называют «жадным» и в чём его слабость? Как её исправляет алгоритм ADD-DEL (раздел 3.3.4)?

**Ваш ответ:** *...*

In [ ]:
# Подсказка к вопросу 1: сколько подмножеств у множества из 30 элементов?
print(2 ** 30)

## Задача 6. Отбор признаков на основе моделей (15 баллов)

### 6.1. L1-регуляризация (раздел 3.4.1)

Обучите логистическую регрессию с L1-штрафом (`penalty="l1"`, `solver="liblinear"`) при значениях `C` из списка. Чем меньше `C`, тем сильнее регуляризация. Для каждого `C` выведите число ненулевых весов и AUC на тесте.

In [ ]:
# ВАШ КОД
for C in [0.01, 0.03, 0.1, 0.3, 1, 10]:
    model = LogisticRegression(penalty="l1", solver="liblinear", C=C)
    model.fit(X_train_s, y_train)
    n_nonzero = ...
    auc_ = ...
    print(f"C={C}: ненулевых весов = {n_nonzero}, AUC = {auc_:.4f}")

Выберите `C = 0.1`, сохраните признаки с ненулевыми весами в `selected_l1`, оцените качество новой модели (обычной логистической регрессии, обученной только на этих признаках) и добавьте в `results`.

In [ ]:
# ВАШ КОД
l1 = LogisticRegression(penalty="l1", solver="liblinear", C=0.1).fit(X_train_s, y_train)
selected_l1 = ...
acc, auc_ = ...
results["L1 (C=0.1)"] = (len(selected_l1), acc, auc_)
print(len(selected_l1), acc, auc_)

### 6.2. Важность признаков в случайном лесе (раздел 3.4.3)

Обучите `RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)` на обучающей выборке. Его атрибут `feature_importances_` — это суммарное уменьшение критерия информативности по всем деревьям. Постройте диаграмму топ-10 признаков, возьмите их и оцените качество логистической регрессии на этих признаках.

In [ ]:
# ВАШ КОД
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE).fit(X_train_s, y_train)
imp_rf = pd.Series(..., index=X_train_s.columns)

...  # barh-диаграмма топ-10
plt.show()

top10_rf = ...
acc, auc_ = ...
results["Случайный лес, топ-10"] = (10, acc, auc_)
print(acc, auc_)

### 6.3. Важность через перемешивание (раздел 3.4.3)

Идея из лекции: «испортим» признак, перемешав его значения, и посмотрим, насколько упадёт качество. В `sklearn` это делает `permutation_importance`. (В лекции перемешивание делается на out-of-bag выборке каждого дерева, здесь для простоты используем тестовую выборку.) Посчитайте важность по перемешиванию для обученного леса на тесте (`n_repeats=10`, `scoring="roc_auc"`) и сравните топ-5 с топ-5 по `feature_importances_`.

In [ ]:
# ВАШ КОД
perm = permutation_importance(rf, X_test_s, y_test, n_repeats=10, scoring="roc_auc", random_state=RANDOM_STATE)
imp_perm = pd.Series(..., index=X_test_s.columns)  # среднее падение качества

print("feature_importances_:", ...)
print("permutation:         ", ...)

In [ ]:
# Проверка задачи 6
assert 1 <= len(selected_l1) < 30 and len(selected_l1) == int((l1.coef_ != 0).sum()), "selected_l1 — признаки с ненулевыми весами"
assert abs(imp_rf.sum() - 1) < 1e-6, "imp_rf — это rf.feature_importances_ (сумма равна 1)"
assert len(top10_rf) == 10 and top10_rf[0] == imp_rf.idxmax(), "top10_rf: 10 самых важных признаков, по убыванию"
assert len(imp_perm) == 30, "imp_perm: среднее падение качества для каждого из 30 признаков"
assert "L1 (C=0.1)" in results and "Случайный лес, топ-10" in results, "Добавьте результаты в results"
print("✅ Задача 6: проверка пройдена")

**Ваш ответ на 6:** *(сколько признаков оставляет L1 при разных `C` и как это влияет на качество? Совпадают ли признаки, выбранные разными методами? Почему у случайного леса важность может «размазываться» между сильно коррелированными признаками?)*

## Задача 7. Метод главных компонент (15 баллов)

Теперь вместо отбора будем **строить новые признаки** — линейные комбинации исходных (разделы 3.5–3.7). Данные уже масштабированы, а для PCA это важно: иначе признаки с большим масштабом перетянут всё на себя.

**7.1.** Обучите `PCA()` на `X_train_s` (со всеми компонентами). Постройте график накопленной доли объяснённой дисперсии и найдите минимальное число компонент, которое сохраняет не менее 90% и не менее 95% дисперсии.

In [ ]:
# ВАШ КОД
pca = PCA().fit(X_train_s)
cum_var = ...  # np.cumsum(pca.explained_variance_ratio_)

plt.plot(range(1, 31), cum_var, marker="o")
plt.axhline(0.95, color="r", ls="--")
plt.xlabel("Число компонент")
plt.ylabel("Накопленная доля дисперсии")
plt.show()

d90 = ...
d95 = ...
print("Для 90%:", d90, " Для 95%:", d95)
print("Доля дисперсии первой компоненты:", ...)

**7.2.** Спроецируйте выборку на 2 главные компоненты и постройте диаграмму рассеяния, раскрасив точки по классу. Разделимы ли классы в двумерном пространстве (ср. с рис. 3.3 лекции)?

In [ ]:
# ВАШ КОД
pca2 = PCA(n_components=2).fit(X_train_s)
Z2 = pca2.transform(X_train_s)

for cls, name, color in [(0, "злокачественная", "crimson"), (1, "доброкачественная", "royalblue")]:
    mask = (y_train.values == cls)
    plt.scatter(..., ..., s=12, alpha=0.6, label=name, color=color)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.show()

**7.3.** Обучите логистическую регрессию на первых `d` главных компонентах для `d` из списка и постройте график зависимости тестового AUC от `d`. Для `d95` добавьте результат в `results`.

In [ ]:
# ВАШ КОД
pca_full = PCA().fit(X_train_s)
Ztr_full = pca_full.transform(X_train_s)
Zte_full = pca_full.transform(X_test_s)

ds = [1, 2, 3, 5, 10, 15, 30]
aucs = []
for d in ds:
    acc, auc_ = evaluate(..., ...)   # первые d столбцов Ztr_full и Zte_full
    aucs.append(auc_)
    print(d, acc, auc_)

plt.plot(ds, aucs, marker="o")
plt.xlabel("Число главных компонент")
plt.ylabel("AUC на тесте")
plt.show()

acc, auc_ = evaluate(Ztr_full[:, :d95], Zte_full[:, :d95])
results[f"PCA ({d95} компонент)"] = (d95, acc, auc_)

**7.4.** Выведите 5 признаков с наибольшими по модулю весами в первой главной компоненте (`pca_full.components_[0]`). Что они значат с точки зрения предметной области?

**Ваш ответ на 7:** *(сколько компонент достаточно для 95% дисперсии? Как PCA соотносится с отбором признаков? Чем новые признаки хуже для интерпретации?)*

In [ ]:
# ВАШ КОД
w1 = pd.Series(pca_full.components_[0], index=X_train_s.columns)
print(...)

In [ ]:
# Проверка задачи 7
assert len(cum_var) == 30 and abs(cum_var[-1] - 1) < 1e-9, "cum_var — накопленная сумма explained_variance_ratio_"
assert 1 <= d90 <= d95 <= 30, "Должно выполняться d90 <= d95"
assert cum_var[d95 - 1] >= 0.95 and (d95 == 1 or cum_var[d95 - 2] < 0.95), "d95 — МИНИМАЛЬНОЕ число компонент для 95%"
assert Z2.shape == (len(X_train_s), 2), "Z2 — проекция обучающей выборки на 2 компоненты"
assert len(aucs) == len(ds) and aucs[-1] > 0.99, "Проверьте обучение на первых d столбцах Ztr_full / Zte_full"
assert f"PCA ({d95} компонент)" in results, "Добавьте результат PCA в results"
assert len(w1) == 30, "w1 — веса первой компоненты по 30 признакам"
print("✅ Задача 7: проверка пройдена")

## Задача 8. Случайные проекции (10 баллов)

В разделе 3.5.2 описан самый простой способ понижения размерности: новые признаки — это **случайные** линейные комбинации исходных. Метод не «учится» на данных и работает очень быстро. Проверим, насколько он хорош на нашем датасете.

**8.1.** Для `d` из списка выполните `GaussianRandomProjection(n_components=d, random_state=RANDOM_STATE)` на обучающей выборке (`fit`), преобразуйте train и test, обучите логистическую регрессию и выведите AUC. Сравните с PCA при том же числе компонент (`aucs` из задачи 7.3 для `d = 2, 5, 10, 15`).

In [ ]:
# ВАШ КОД
for d in [2, 5, 10, 15]:
    rp = GaussianRandomProjection(n_components=d, random_state=RANDOM_STATE).fit(X_train_s)
    Ztr = ...
    Zte = ...
    acc, auc_ = evaluate(Ztr, Zte)
    print(f"d={d}: случайные проекции AUC = {auc_}, PCA AUC = {aucs[ds.index(d)]}")
    if d == 10:
        results["Случайные проекции (10)"] = (10, acc, auc_)

**8.2.** Проверим сохранение расстояний. Возьмите первые 100 объектов обучающей выборки, посчитайте попарные расстояния (`pdist`) до и после проекции в `d = 15` и найдите корреляцию между двумя наборами расстояний.

In [ ]:
# ВАШ КОД
sub = X_train_s.values[:100]
rp15 = GaussianRandomProjection(n_components=15, random_state=RANDOM_STATE).fit(X_train_s)
dist_before = pdist(sub)
dist_after = ...
print("Корреляция расстояний:", ...)

In [ ]:
# Проверка задачи 8
assert Ztr.shape == (len(X_train_s), 15) and Zte.shape == (len(X_test_s), 15), "В цикле нужно преобразовать train и test через rp.transform"
assert len(dist_after) == len(dist_before) == 100 * 99 // 2, "dist_after: pdist от проекции тех же 100 объектов"
assert np.corrcoef(dist_before, dist_after)[0, 1] > 0.8, "Расстояния должны примерно сохраняться"
assert "Случайные проекции (10)" in results, "Добавьте результат для d=10 в results"
print("✅ Задача 8: проверка пройдена")

**Ваш ответ на 8:** *(Почему случайные проекции лучше подходят для задач с тысячами признаков, например для текстов? Чем они проигрывают PCA?)*

## Задача 9. Итоговое сравнение и выводы (5 баллов)

**9.1.** Выведите таблицу `results` (число признаков, accuracy, AUC) для всех методов, отсортированную по AUC.

In [ ]:
# ВАШ КОД
summary = pd.DataFrame(results, index=["Число признаков", "Accuracy", "AUC"]).T
summary_sorted = ...
display(summary_sorted)

In [ ]:
# Проверка задачи 9
assert summary_sorted.shape == (7, 3), "В таблице должно быть 7 методов и 3 столбца. Все ли результаты добавлены в results?"
assert summary_sorted["AUC"].is_monotonic_decreasing, "Таблица должна быть отсортирована по AUC по убыванию"
print("✅ Задача 9: проверка пройдена. Остались только текстовые ответы!")

**9.2.** Напишите краткие выводы (5–7 предложений):
1. Удалось ли сократить число признаков без заметной потери качества? Какой метод оказался лучшим по соотношению «качество / число признаков»?
2. В каких ситуациях вы бы выбрали **отбор** признаков, а в каких — **понижение размерности**? (Учтите интерпретируемость, скорость работы модели — раздел 3.1.1, и избыточность данных.)
3. Какое ограничение у выводов, сделанных на одном разбиении train/test?

**Ваши выводы:** *...*